##imports an other

In [1]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

In [ ]:
!pip install -q transformers accelerate

In [ ]:
%pip install scikit-learn transformers gradio fastai

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

##data


In [7]:
df1 = pd.read_csv("./data/PDS_FinalProject/department-v2.csv")
df2 = pd.read_csv("./data/PDS_FinalProject/seniority-v2.csv")

FileNotFoundError: [Errno 2] No such file or directory: './data/PDS_FinalProject/department-v2.csv'

In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10145 entries, 0 to 10144
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    10145 non-null  object
 1   label   10145 non-null  object
dtypes: object(2)
memory usage: 158.6+ KB


In [ ]:
df1['label'].unique()

array(['Marketing', 'Project Management', 'Administrative',
       'Business Development', 'Consulting', 'Human Resources',
       'Information Technology', 'Other', 'Purchasing', 'Sales',
       'Customer Support'], dtype=object)

In [ ]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9428 entries, 0 to 9427
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    9428 non-null   object
 1   label   9428 non-null   object
dtypes: object(2)
memory usage: 147.4+ KB


In [ ]:
df2['label'].unique()

array(['Junior', 'Senior', 'Lead', 'Management', 'Director'], dtype=object)

In [ ]:
df2['label'].value_counts()

,count
label,
Senior,3733
Lead,3546
Director,984
Management,756
Junior,409


In [ ]:
df_test_notannotated = pd.read_json("/content//PDS_FinalProject/linkedin-cvs-not-annotated.json")
df_test_annotated = pd.read_json("/content/PDS_FinalProject/linkedin-cvs-annotated.json")


In [ ]:
def get_json(data):
    return json.loads(Path(data).read_text(encoding="utf-8"))

def workers_positions_to_csv(data, csv_file="workers_positions.csv"):
    people = get_json(data)
    rows = []

    for person_idx, jobs in enumerate(people, start=1):
        jobs_sorted = sorted(jobs, key=lambda d: (d.get("startDate") is None, d.get("startDate")))
        for job_idx, j in enumerate(jobs_sorted, start=1):
            rows.append({
                "person_id": person_idx,
                "job_no": job_idx,
                "organization": j.get("organization"),
                "linkedin": j.get("linkedin"),
                "position": j.get("position"),
                "startDate": j.get("startDate"),
                "endDate": j.get("endDate"),
                "status": j.get("status"),
                "department": j.get("department"),
                "seniority": j.get("seniority"),
            })

    df = pd.DataFrame(rows, columns=[
        "person_id","job_no","organization","linkedin","position",
        "startDate","endDate","status","department","seniority"
    ])
    df["person_id"] = df["person_id"].astype(int)
    df["job_no"] = df["job_no"].astype(int)
    return df, csv_file

In [ ]:
df_t_annotated, path = workers_positions_to_csv("/content/PDS_FinalProject/linkedin-cvs-annotated.json")
df_t_notannotated, path = workers_positions_to_csv("/content/PDS_FinalProject/linkedin-cvs-not-annotated.json")

In [ ]:
df_t_annotated["department"].unique()

array(['Other', 'Customer Support', 'Information Technology', 'Sales',
       'Project Management', 'Consulting', 'Administrative',
       'Business Development', 'Marketing', 'Human Resources',
       'Purchasing'], dtype=object)

In [ ]:
df1["label"].value_counts()

,count
label,
Marketing,4295
Sales,3328
Information Technology,1305
Business Development,620
Project Management,201
Consulting,167
Administrative,83
Other,42
Purchasing,40


In [ ]:
df_t_annotated["department"].value_counts()

,count
department,
Other,1252
Information Technology,312
Sales,219
Consulting,195
Project Management,175
Marketing,133
Administrative,84
Business Development,78
Purchasing,72


In [ ]:
df_t_annotated

,person_id,job_no,organization,linkedin,position,startDate,endDate,status,department,seniority
0,1,1,Nagel Car Group,,Buchhalterin,2000-05,2019-06,INACTIVE,Other,Professional
1,1,2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management
2,1,3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,None,ACTIVE,Other,Professional
3,1,4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,None,ACTIVE,Other,Management
4,1,5,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management
...,...,...,...,...,...,...,...,...,...,...
2633,608,2,Malteser Hilfsdienst,https://www.linkedin.com/company/malteser-hilf...,Präsidium,1999,None,ACTIVE,Other,Management
2634,608,3,FORESTA Management,,Geschäftsführer,2010-01,None,ACTIVE,Other,Management
2635,608,4,Bistum,,Justitiar,2014-03,None,ACTIVE,Other,Professional
2636,608,5,Malteserorden,,Kanzler der deutschen Assoziation,2015-01,None,ACTIVE,Other,Management


In [ ]:
df_t_annotated[df_t_annotated["person_id"] == 2]

,person_id,job_no,organization,linkedin,position,startDate,endDate,status,department,seniority
6,2,1,MicroAge College Station,https://www.linkedin.com/company/avinext,Support Specialist,2002-10,2006-04,INACTIVE,Customer Support,Professional
7,2,2,Texas A&M Forest Service,https://www.linkedin.com/company/texasforestse...,Network Systems Administrator II,2006-04,2010-02,INACTIVE,Information Technology,Professional
8,2,3,Texas A&M University at Qatar,,Senior Systems Administrator,2010-02,2012-12,INACTIVE,Information Technology,Senior
9,2,4,Texas A&M University at Qatar,,Lead Network Engineer,2012-12,2015-06,INACTIVE,Information Technology,Lead
10,2,5,Texas A&M University-Corpus Christi,,Infrastructure Administrator II,2015-06,2017-02,INACTIVE,Information Technology,Professional
11,2,6,Texas A&M University-Corpus Christi,,Manager of Network Services,2017-02,2019-07,INACTIVE,Information Technology,Professional
12,2,7,Computer Solutions,https://www.linkedin.com/company/computer-solu...,Senior Network Engineer,2019-07,2024-03,INACTIVE,Information Technology,Senior
13,2,8,Computer Solutions,https://www.linkedin.com/company/computer-solu...,Solutions Architect,2024-03,None,ACTIVE,Information Technology,Professional


In [ ]:
df_t_annotated["seniority"].unique()

array(['Professional', 'Management', 'Senior', 'Lead', 'Director',
       'Junior'], dtype=object)

In [ ]:
df_t_annotated["seniority"].value_counts()

,count
seniority,
Professional,1219
Lead,460
Management,418
Junior,230
Senior,170
Director,141


##1st Model

###depa

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    df1["text"], df1["label"], test_size=0.2, random_state=42, stratify=df1["label"]
)

pipe0 = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1,2), min_df=3)),
    ("clf", LinearSVC())
])

pipe0.fit(X_train, y_train)
y_pred = pipe0.predict(X_val)
print("accuracy:", accuracy_score(y_val, y_pred))

accuracy: 0.9743716116313454


In [ ]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, strip_accents="unicode",
                              ngram_range=(1, 2), min_df=3)),
    ("clf", LinearSVC())
])
pipe.fit(df1["text"], df1["label"])

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(min_df=3, ngram_range=(1, 2),
                                 strip_accents='unicode')),
                ('clf', LinearSVC())])

In [ ]:
df_active = df_t_annotated[df_t_annotated["status"] == "ACTIVE"]


y_true = df_active["department"]
y_pred = pipe.predict(df_active["position"])

In [ ]:
print(f"ACTIVE rows: {len(df_active)}")
print("accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, digits=3))

ACTIVE rows: 623
accuracy: 0.2231139646869984
                        precision    recall  f1-score   support

        Administrative      0.333     0.071     0.118        14
  Business Development      0.375     0.300     0.333        20
            Consulting      0.633     0.487     0.551        39
      Customer Support      1.000     0.167     0.286         6
       Human Resources      0.727     0.500     0.593        16
Information Technology      0.262     0.548     0.354        62
             Marketing      0.060     0.682     0.111        22
                 Other      0.000     0.000     0.000       344
    Project Management      0.396     0.487     0.437        39
            Purchasing      1.000     0.400     0.571        15
                 Sales      0.236     0.652     0.347        46

              accuracy                          0.223       623
             macro avg      0.457     0.390     0.336       623
          weighted avg      0.182     0.223     0.172   

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
X_train_text = df1["text"]

df_active = df_t_annotated[df_t_annotated["status"] == "ACTIVE"]
X_eval_text = df_active["position"]
y_eval = df_active["department"]

vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 6),
    strip_accents="unicode",
    min_df=2,
    max_df=0.995,
    sublinear_tf=True,
)

vec.fit(X_train_text)

X_train = vec.transform(X_train_text)
X_eval  = vec.transform(X_eval_text)

clf = LogisticRegression(
    max_iter=4000,
    C=2.0,
    class_weight="balanced",
    n_jobs=-1
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_eval)
print(f"ACTIVE rows: {len(y_eval)}")
print("accuracy:", accuracy_score(y_eval, y_pred))
print(classification_report(y_eval, y_pred, digits=3))

ACTIVE rows: 623
accuracy: 0.26484751203852325
                        precision    recall  f1-score   support

        Administrative      0.100     0.214     0.136        14
  Business Development      0.375     0.300     0.333        20
            Consulting      0.905     0.487     0.633        39
      Customer Support      0.500     0.167     0.250         6
       Human Resources      0.643     0.562     0.600        16
Information Technology      0.121     0.839     0.212        62
             Marketing      0.562     0.409     0.474        22
                 Other      0.429     0.009     0.017       344
    Project Management      0.686     0.615     0.649        39
            Purchasing      1.000     0.400     0.571        15
                 Sales      0.688     0.717     0.702        46

              accuracy                          0.265       623
             macro avg      0.546     0.429     0.416       623
          weighted avg      0.479     0.265     0.225  

In [ ]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        strip_accents="unicode",
        min_df=2,
        max_df=0.995,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(
        max_iter=4000,
        C=2.0,
        class_weight="balanced",
        n_jobs=-1,
    )),
])

pipe.fit(df1["text"], df1["label"])

df_active = df_t_annotated[df_t_annotated["status"] == "ACTIVE"]
y_true = df_active["department"]
y_pred = pipe.predict(df_active["position"])

print(f"ACTIVE rows: {len(df_active)}")
print("accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, digits=3))

ACTIVE rows: 623
accuracy: 0.2680577849117175
                        precision    recall  f1-score   support

        Administrative      0.115     0.214     0.150        14
  Business Development      0.375     0.300     0.333        20
            Consulting      0.909     0.513     0.656        39
      Customer Support      0.500     0.167     0.250         6
       Human Resources      0.692     0.562     0.621        16
Information Technology      0.125     0.855     0.218        62
             Marketing      0.391     0.409     0.400        22
                 Other      0.429     0.009     0.017       344
    Project Management      0.706     0.615     0.658        39
            Purchasing      1.000     0.400     0.571        15
                 Sales      0.673     0.717     0.695        46

              accuracy                          0.268       623
             macro avg      0.538     0.433     0.415       623
          weighted avg      0.475     0.268     0.225   

In [ ]:
X_eval_text2 = df_t_annotated["position"]
y_eval2 = df_t_annotated["department"]
X_eval2  = vec.transform(X_eval_text2)
y2_pred = clf.predict(X_eval2)
print(f"ALL rows: {len(y_eval2)}")
print("accuracy:", accuracy_score(y_eval2, y2_pred))
print(classification_report(y_eval2, y2_pred, digits=3))

ALL rows: 2638
accuracy: 0.3290371493555724
                        precision    recall  f1-score   support

        Administrative      0.139     0.167     0.151        84
  Business Development      0.389     0.449     0.417        78
            Consulting      0.833     0.538     0.654       195
      Customer Support      0.739     0.354     0.479        48
       Human Resources      0.796     0.557     0.655        70
Information Technology      0.162     0.904     0.275       312
             Marketing      0.678     0.444     0.536       133
                 Other      0.645     0.016     0.031      1252
    Project Management      0.686     0.623     0.653       175
            Purchasing      0.926     0.347     0.505        72
                 Sales      0.787     0.744     0.765       219

              accuracy                          0.329      2638
             macro avg      0.616     0.468     0.466      2638
          weighted avg      0.608     0.329     0.287     

###seni

In [ ]:
pipe2 = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        strip_accents="unicode",
        min_df=2,
        max_df=0.995,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(
        max_iter=4000,
        C=2.0,
        class_weight="balanced",
        n_jobs=-1,
    )),
])

pipe2.fit(df2["text"], df2["label"])

df_active = df_t_annotated[df_t_annotated["status"] == "ACTIVE"]
y_true = df_active["seniority"]
y_pred = pipe2.predict(df_active["position"])

print(f"ACTIVE rows: {len(df_active)}")
print("accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, digits=3))

ACTIVE rows: 623
accuracy: 0.4943820224719101
              precision    recall  f1-score   support

    Director      0.552     0.941     0.696        34
      Junior      0.222     0.333     0.267        12
        Lead      0.561     0.704     0.624       125
  Management      0.670     0.760     0.712       192
Professional      0.000     0.000     0.000       216
      Senior      0.221     0.864     0.352        44

    accuracy                          0.494       623
   macro avg      0.371     0.600     0.442       623
weighted avg      0.369     0.494     0.413       623



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
